# Gemini version of 04d — within-subject utterance similarity

Parallel to `04d_talk_generate_cossim_toself`, using the full-time utterance
embeddings from `03g_talk_generate_gemini_embedding`. Same 5-way (own category
excluded) softmax logic as `04cg` (drawings)

Output: `data/preprocessed/gemini/talk_cossim_toself.csv`, same `utterance_cossim_{cat}`
/ `utterance_probability_{cat}` columns as the CLIP version.

In [1]:
import ast
from pathlib import Path

import numpy as np
import pandas as pd

CATEGORIES = ["crab", "octopus", "seahorse", "shark", "turtle", "whale"]

EMBED_CSV = "../data/embeddings/gemini/utterance_gemini_embeddings.csv"
OUT_CSV = "../data/preprocessed/gemini/talk_cossim_toself.csv"

FLAGS_CSV = "../data/tidy/utterances.csv"  # interference flags

# Interference trials are excluded throughout, so they enter neither the comparison set nor the output
_flags = pd.read_csv(FLAGS_CSV)
INTERFERENCE = set(zip(_flags.loc[_flags["utterance_interference"] == 1, "participant_id"].str.strip(),
                       _flags.loc[_flags["utterance_interference"] == 1, "target_category"].str.strip().str.lower()))

In [2]:
def parse_embedding(x):
    if isinstance(x, np.ndarray):
        return x
    if isinstance(x, list):
        return np.array(x, dtype=float)
    if pd.isna(x):
        return np.nan
    return np.array(ast.literal_eval(x), dtype=float)


def cosine_sim_to_matrix(query, matrix):
    """cosine(query, each row of matrix) -> 1D array, len == matrix.shape[0]."""
    q = query / np.linalg.norm(query)
    m = matrix / np.linalg.norm(matrix, axis=1, keepdims=True)
    return m @ q


def softmax(x):
    x = x - np.max(x)
    e = np.exp(x)
    return e / e.sum()

In [3]:
transcript_df = pd.read_csv(EMBED_CSV)
transcript_df["participant_id"] = transcript_df["participant_id"].astype(str).str.strip()
transcript_df["target_category"] = transcript_df["target_category"].astype(str).str.lower().str.strip()
transcript_df = transcript_df[[k not in INTERFERENCE for k in zip(transcript_df["participant_id"], transcript_df["target_category"])]].copy()
transcript_df["utterance_full_embedding"] = transcript_df["utterance_full_embedding"].apply(parse_embedding)

print(transcript_df.shape)
print("valid full embeddings:", transcript_df["utterance_full_embedding"].apply(lambda x: isinstance(x, np.ndarray)).sum())

(631, 8)
valid full embeddings: 538


## Softmax temperature

Same computation as `04cg`'s diagnostic, run here on utterance embeddings
specifically: median within-participant cosine spread, scaled against CLIP's
typical ~0.55 spread.

In [4]:
emb = {}
for _, r in transcript_df.iterrows():
    e = r["utterance_full_embedding"]
    if isinstance(e, np.ndarray):
        emb[(r["participant_id"], r["target_category"])] = e

spreads = []
for pid in transcript_df["participant_id"].unique():
    vecs = {c: emb[(pid, c)] for c in CATEGORIES if (pid, c) in emb}
    if len(vecs) < 3:
        continue
    for c_query in vecs:
        others = [c for c in vecs if c != c_query]
        sims = [float(np.dot(vecs[c_query], vecs[c]) /
                       (np.linalg.norm(vecs[c_query]) * np.linalg.norm(vecs[c])))
                for c in others]
        spreads.append(max(sims) - min(sims))

spreads = np.array(spreads)
CLIP_REFERENCE_SPREAD = 0.55  # typical within-participant range in the CLIP branch
median_spread = float(np.median(spreads))
TEMPERATURE = median_spread / CLIP_REFERENCE_SPREAD

print(f"within-participant cosine spread (n={len(spreads)}): median {median_spread:.4f}")
print(f"softmax temperature: {TEMPERATURE:.4f}  (i.e. softmax(sims / {TEMPERATURE:.4f}))")

within-participant cosine spread (n=528): median 0.0719
softmax temperature: 0.1307  (i.e. softmax(sims / 0.1307))


In [5]:
def add_within_subject_utterance_similarities(
    df,
    temperature=1.0,
    embedding_col="utterance_full_embedding",
    participant_col="participant_id",
    category_col="target_category",
    categories=None,
):
    if categories is None:
        categories = CATEGORIES

    df = df.copy()
    for cat in categories:
        df[f"utterance_cossim_{cat}"] = np.nan
        df[f"utterance_probability_{cat}"] = np.nan

    for pid, g in df.groupby(participant_col):
        cat_to_emb = {}
        for cat in categories:
            embs = [e for e in g.loc[g[category_col] == cat, embedding_col] if isinstance(e, np.ndarray)]
            if embs:
                cat_to_emb[cat] = np.mean(np.stack(embs), axis=0)

        for idx, row in g.iterrows():
            query_emb = row[embedding_col]
            if not isinstance(query_emb, np.ndarray):
                continue

            row_cat = row[category_col]
            comparison_cats = [c for c in categories if c != row_cat and c in cat_to_emb]
            if not comparison_cats:
                continue

            category_matrix = np.stack([cat_to_emb[c] for c in comparison_cats])
            sims = cosine_sim_to_matrix(query_emb, category_matrix)
            probs = softmax(sims / temperature)

            for cat, sim, prob in zip(comparison_cats, sims, probs):
                df.loc[idx, f"utterance_cossim_{cat}"] = sim
                df.loc[idx, f"utterance_probability_{cat}"] = prob

    return df

In [6]:
transcript_df = add_within_subject_utterance_similarities(transcript_df, temperature=TEMPERATURE)

prob_cols = [f"utterance_probability_{c}" for c in CATEGORIES]
transcript_df["prob_sum"] = transcript_df[prob_cols].sum(axis=1, min_count=1)
transcript_df["prob_sum"].describe()

count    5.340000e+02
mean     1.000000e+00
std      8.412156e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: prob_sum, dtype: float64

In [7]:
cols_to_drop = [
    "utterance_masked", "utterance_half_masked_filtered",
    "utterance_full_embedding", "utterance_half_embedding", "prob_sum",
]
result = transcript_df.drop(columns=[c for c in cols_to_drop if c in transcript_df.columns])

Path(OUT_CSV).parent.mkdir(parents=True, exist_ok=True)
result.to_csv(OUT_CSV, index=False)
print(f"Saved {result.shape[0]} rows -> {OUT_CSV}")
print(result.columns.tolist())

Saved 631 rows -> ../data/preprocessed/gemini/talk_cossim_toself.csv
['participant_id', 'age_months', 'target_category', 'is_adult', 'utterance_cossim_crab', 'utterance_probability_crab', 'utterance_cossim_octopus', 'utterance_probability_octopus', 'utterance_cossim_seahorse', 'utterance_probability_seahorse', 'utterance_cossim_shark', 'utterance_probability_shark', 'utterance_cossim_turtle', 'utterance_probability_turtle', 'utterance_cossim_whale', 'utterance_probability_whale']
